# W10 — Assignment notebook

**Tema:** Particionamiento de datos + Partition Pruning  
**Dataset:** `silver_planet_v3` (6 087 planetas — NASA Exoplanet Archive)

## Setup

In [ ]:
from pathlib import Path
import duckdb

# Rutas del proyecto
ROOT     = Path(".").resolve()
DB_PATH  = ROOT / "data" / "exoplanets.duckdb"
RAW_CSV  = ROOT / "data" / "raw" / "pscomppars.csv"
PART_DIR = ROOT / "data" / "partitioned"

def quoted(p: Path) -> str:
    """Devuelve la ruta como string SQL escapado."""
    return "'" + p.resolve().as_posix().replace("'", "''") + "'"

# Conexión y vista del CSV crudo
con = duckdb.connect(str(DB_PATH))
con.execute("DROP VIEW IF EXISTS raw_ps")
con.execute(f"CREATE VIEW raw_ps AS SELECT * FROM read_csv_auto({quoted(RAW_CSV)})")

PART_DIR.mkdir(parents=True, exist_ok=True)

n_rows = con.sql("SELECT COUNT(*) FROM silver_planet_v3").fetchone()[0]
print(f"Setup OK — filas en silver_planet_v3: {n_rows:,}")

## 1. Evidencia de particionamiento

Se tomó la tabla `silver_planet_v3` y se escribió una partición Parquet independiente para cada valor de `disc_era`. Cada carpeta sigue la convención `disc_era=<valor>/data.parquet` para que motores como DuckDB o Spark puedan inferir el esquema de partición automáticamente.

In [ ]:
ERAS = ["pre-2000", "2000s", "2010s", "2020s", "unknown"]

for era in ERAS:
    dest = PART_DIR / f"disc_era={era}"
    dest.mkdir(exist_ok=True)
    con.execute(f"""
        COPY (
            SELECT * FROM silver_planet_v3
            WHERE disc_era = '{era}'
        )
        TO '{dest}/data.parquet' (FORMAT PARQUET)
    """)

print("Particiones generadas:\n")
for parquet_file in sorted(PART_DIR.rglob("*.parquet")):
    size_bytes = parquet_file.stat().st_size
    rel_path   = parquet_file.relative_to(PART_DIR)
    print(f"  {rel_path}  →  {size_bytes:,} bytes  ({size_bytes / 1024:.1f} KB)")

## 2. Número de archivos generados

In [ ]:
parquet_files = sorted(PART_DIR.rglob("*.parquet"))

print(f"Total archivos Parquet generados: {len(parquet_files)}\n")
for f in parquet_files:
    print(f"  {f.name}  en  {f.parent.name}/")

## 3. Resumen por partición

In [ ]:
TOTAL_ROWS = 6087

header = f"{'disc_era':<12}  {'rows':>6}  {'size_KB':>8}  {'pct_rows':>9}"
print(header)
print("-" * len(header))

total_rows = 0
for era in ERAS:
    f      = PART_DIR / f"disc_era={era}" / "data.parquet"
    rows   = con.sql(f"SELECT COUNT(*) FROM '{f}'").fetchone()[0]
    size   = f.stat().st_size / 1024
    pct    = rows / TOTAL_ROWS * 100
    print(f"  {era:<12}  {rows:>6}  {size:>7.1f}  {pct:>8.2f}%")
    total_rows += rows

print("-" * len(header))
print(f"  {'TOTAL':<12}  {total_rows:>6}")

## 4. Evidencia de pruning

Para verificar que el pruning funciona, se ejecuta `EXPLAIN ANALYZE` sobre dos fuentes distintas:
la tabla completa y el archivo Parquet de una sola era. El plan resultante debe mostrar una diferencia clara en la cantidad de filas procesadas.

In [ ]:
PARTITION_2010S = PART_DIR / "disc_era=2010s" / "data.parquet"

QUERY_AGG = """
    SELECT discoverymethod_canon, COUNT(*) AS n
    FROM {source}
    GROUP BY discoverymethod_canon
    ORDER BY n DESC
"""

print("=== SIN pruning — TABLE_SCAN sobre silver_planet_v3 (6 087 filas) ===")
con.sql(f"EXPLAIN ANALYZE {QUERY_AGG.format(source='silver_planet_v3')}").show()

print("=== CON pruning — READ_PARQUET sobre disc_era=2010s (3 683 filas) ===")
con.sql(f"EXPLAIN ANALYZE {QUERY_AGG.format(source=repr(str(PARTITION_2010S)))}").show()

## 5. Guardado de EXPLAIN ANALYZE como texto

In [ ]:
out_txt = ROOT / "docs" / "w10b_explain_analyze_pruning.txt"
out_txt.parent.mkdir(exist_ok=True)

def explain_to_lines(query: str) -> list[str]:
    rows = con.sql(f"EXPLAIN ANALYZE {query}").fetchall()
    return [row[1] + "\n" for row in rows]

sections = [
    ("SIN PRUNING — silver_planet_v3 (6 087 filas)",
     QUERY_AGG.format(source="silver_planet_v3")),
    ("CON PRUNING — disc_era=2010s (3 683 filas)",
     QUERY_AGG.format(source=repr(str(PARTITION_2010S)))),
]

output_lines = []
for title, query in sections:
    output_lines.append(f"=== {title} ===\n")
    output_lines.extend(explain_to_lines(query))
    output_lines.append("\n")

out_txt.write_text("".join(output_lines))
print(f"Guardado en: {out_txt}")

## 6. Explicación del filtro de pruning usado

El predicado aplicado fue **`disc_era = '2010s'`**, que acota el análisis a los planetas detectados durante la década 2010–2019.

Dado que el archivo `disc_era=2010s/data.parquet` contiene únicamente filas de esa era, el motor no necesita abrir los otros cuatro archivos. Esto se refleja directamente en el plan de ejecución:

| Escenario | Nodo del plan | Filas procesadas |
|-----------|---------------|------------------|
| Sin pruning | `TABLE_SCAN` | 6 087 |
| Con pruning | `READ_PARQUET · Files Read: 1` | 3 683 |

El motor evitó leer el **39.5 %** del dataset (2 404 filas) simplemente eligiendo el archivo correcto.

## 7. Decisión de partición

In [ ]:
print("Distribución por disc_era (columna de partición elegida):")
con.sql("""
    SELECT
        disc_era,
        COUNT(*) AS n,
        ROUND(COUNT(*) * 100.0 / 6087, 2) AS pct
    FROM silver_planet_v3
    GROUP BY disc_era
    ORDER BY disc_era
""").show()

print("\nAlternativa evaluada — distribución por discoverymethod_canon:")
con.sql("""
    SELECT
        discoverymethod_canon,
        COUNT(*) AS n,
        ROUND(COUNT(*) * 100.0 / 6087, 2) AS pct
    FROM silver_planet_v3
    GROUP BY discoverymethod_canon
    ORDER BY n DESC
""").show()

## 8. ¿Por qué `disc_era` sí?

La elección de `disc_era` como columna de partición se sostiene en cuatro argumentos:

1. **Cardinalidad controlada:** con exactamente 5 valores posibles se producen 5 archivos. No hay riesgo de generar cientos de particiones diminutas.
2. **Compatibilidad con los patrones de consulta:** los estudios sobre tendencias de descubrimiento casi siempre delimitan el período de análisis, por lo que el filtro por era aparece de forma natural en las queries.
3. **Estabilidad de los datos:** el año de descubrimiento de un exoplaneta no se corrige retroactivamente, así que las particiones no requieren reescritura.
4. **Balance aceptable:** aunque la partición `2010s` concentra el 60 % de las filas, las demás no están vacías. El skew existe pero no invalida la estrategia para este volumen de datos.

> **Nota:** en un entorno de producción con millones de registros, la partición `2010s` podría subdividirse por método de detección para equilibrar la carga.

## 9. ¿Qué otra columna evaluaría?

El siguiente candidato natural es **`discoverymethod_canon`**, pero los números de la sección 10 lo descartan.

Una alternativa más interesante sería una **jerarquía de dos niveles**: particionar primero por `disc_era` y luego por `discoverymethod_canon` dentro de cada era. Esto permitiría responder preguntas como *¿cuántos planetas de tránsito se confirmaron en los 2010s?* con un único archivo leído. Sin embargo, 5 eras × 11 métodos = 55 combinaciones con apenas 6 087 filas en total implicaría que la mayoría de esas combinaciones producirían archivos de decenas de filas, lo cual contrarresta cualquier ganancia de pruning.

## 10. Riesgo de *small files*

In [ ]:
SMALL_FILE_THRESHOLD = 50  # filas mínimas para considerar una partición viable

print("Riesgo de small files — partición hipotética por discoverymethod_canon:\n")
con.sql(f"""
    SELECT
        discoverymethod_canon,
        COUNT(*) AS n_rows,
        ROUND(COUNT(*) * 100.0 / 6087, 2) AS pct,
        CASE
            WHEN COUNT(*) < {SMALL_FILE_THRESHOLD} THEN '⚠ SMALL FILE'
            ELSE 'OK'
        END AS riesgo
    FROM silver_planet_v3
    GROUP BY discoverymethod_canon
    ORDER BY n_rows ASC
""").show()

Con `discoverymethod_canon` como columna de partición, **7 de los 11 métodos** generarían particiones de menos de 50 filas. En plataformas distribuidas o en la nube (Spark, Athena, BigQuery), abrir un archivo tiene un costo fijo de latencia y metadata que no depende de cuántas filas contiene. Un archivo de 3 filas consume casi tanto tiempo de apertura como uno de 3 000.

Con `disc_era` el problema es mucho más acotado: solo `unknown` (1 fila) y `pre-2000` (30 filas) quedan por debajo del umbral, y representan menos del 0.5 % del total de registros.

## 11. Reflexión breve

El ejercicio deja claro que particionar no es una optimización genérica sino una **decisión de diseño** que depende del uso real del dato. Antes de particionar conviene responder:

- *¿Por qué columna filtran mis queries con más frecuencia?*
- *¿Cuántos valores distintos tiene esa columna?*
- *¿Cuántas filas quedarán por partición? ¿Justifican el overhead de un archivo separado?*

En este dataset, `disc_era` es la única columna que responde bien las tres preguntas. Si el volumen creciera sustancialmente, la respuesta óptima podría cambiar y valdría la pena hacer benchmarking antes de decidir.

## 12. ¿Cuándo particionar ayuda?

El particionamiento aporta valor real en estos escenarios:

- **Volumen alto:** a partir de varios GB, el costo de escanear datos irrelevantes se vuelve perceptible. El pruning lo elimina.
- **Filtros predecibles:** si el 80 % de las queries incluye `WHERE fecha_año = ...` o `WHERE región = ...`, esa columna es candidata directa.
- **Cardinalidad baja y estable:** pocas particiones, bien dimensionadas, con valores que no cambian con el tiempo.
- **Actualizaciones parciales:** si solo se necesita reescribir el último mes sin tocar el histórico, las particiones por fecha lo hacen trivial.
- **Sistemas que cobran por escaneo:** en Athena o BigQuery cada byte leído tiene un costo. El pruning reduce la factura directamente.

## 13. ¿Cuándo particionar empeora el diseño?

Hay situaciones en que particionar introduce más problemas de los que resuelve:

- **Dataset pequeño:** si todo cabe cómodamente en memoria, el overhead de abrir múltiples archivos supera con creces el tiempo ahorrado por el pruning.
- **Alta cardinalidad en la columna de partición:** particionar por un campo con miles de valores únicos (como un ID de usuario o un nombre de estrella) genera miles de archivos minúsculos.
- **Skew severo:** si una sola partición concentra el 95 % de los datos, el motor sigue leyendo casi todo y las demás particiones son overhead puro.
- **Queries que ignoran la columna de partición:** si ninguna consulta filtra por esa columna, el pruning nunca se activa y se paga el costo de la fragmentación sin ningún beneficio.
- **Partición compuesta con demasiados niveles:** combinar tres o más columnas puede generar una explosión de directorios vacíos o con una sola fila, degradando el rendimiento de los listados de metadata.